# Variance Anomaly Detection - ML Pipeline

**Reconciliation Management System**

This notebook demonstrates an end-to-end ML workflow for detecting anomalous variances in financial reconciliations:

1. **EDA** - Exploratory Data Analysis
2. **Feature Engineering** - Using the dbt feature store
3. **Model Training** - Multiple algorithms with hyperparameter tuning
4. **Experiment Tracking** - Snowflake ML experiment logging
5. **Model Registry** - Version management and promotion
6. **Best Model Selection** - Automatic promotion based on metrics

---

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

session = get_active_session()
print(f"Session established: {session.get_current_database()}.{session.get_current_schema()}")

---
## 1. Exploratory Data Analysis (EDA)

Let's understand our feature store data before building models.

In [ ]:
%%sql -r data_summary
SELECT 
    COUNT(*) as total_records,
    COUNT(DISTINCT assignment_id) as unique_assignments,
    COUNT(DISTINCT entity_id) as unique_entities,
    COUNT(DISTINCT period_id) as unique_periods,
    SUM(is_anomaly_label) as labeled_anomalies,
    ROUND(100.0 * SUM(is_anomaly_label) / COUNT(*), 2) as anomaly_pct
FROM COCO_LIVE_DB.PUBLIC.MART_ANOMALY_FEATURES

In [ ]:
feature_df = session.table('COCO_LIVE_DB.PUBLIC.MART_ANOMALY_FEATURES').to_pandas()
print(f"Dataset shape: {feature_df.shape}")
print(f"\nColumn types:\n{feature_df.dtypes}")

In [ ]:
numeric_cols = feature_df.select_dtypes(include=[np.number]).columns.tolist()
exclude_cols = ['ASSIGNMENT_ID', 'ENTITY_ID', 'PERIOD_ID', 'PERIOD_YEAR', 'PERIOD_QUARTER', 'PERIOD_MONTH_NUM', 'IS_ANOMALY_LABEL']
feature_cols = [c for c in numeric_cols if c not in exclude_cols]

stats_df = feature_df[feature_cols].describe().T
stats_df['missing_pct'] = (feature_df[feature_cols].isnull().sum() / len(feature_df) * 100).values
stats_df = stats_df.round(2)
print("Feature Statistics:")
stats_df

In [ ]:
fig = make_subplots(rows=2, cols=2, subplot_titles=(
    'Distribution of Variance Z-Scores',
    'Distribution of GL Balance Z-Scores', 
    'Anomaly Label Distribution',
    'Period Variance Distribution'
))

fig.add_trace(go.Histogram(x=feature_df['VARIANCE_ZSCORE'], name='Variance Z-Score', marker_color='#3498db', nbinsx=50), row=1, col=1)
fig.add_trace(go.Histogram(x=feature_df['GL_BALANCE_ZSCORE'], name='GL Z-Score', marker_color='#e74c3c', nbinsx=50), row=1, col=2)

anomaly_counts = feature_df['IS_ANOMALY_LABEL'].value_counts()
fig.add_trace(go.Bar(x=['Normal', 'Anomaly'], y=[anomaly_counts.get(0, 0), anomaly_counts.get(1, 0)], marker_color=['#28a745', '#dc3545']), row=2, col=1)

fig.add_trace(go.Histogram(x=np.log1p(feature_df['PERIOD_ABS_VARIANCE'].abs()), name='Log Variance', marker_color='#9b59b6', nbinsx=50), row=2, col=2)

fig.update_layout(height=600, title_text='EDA: Feature Distributions', showlegend=False)
fig.show()

In [ ]:
correlation_features = ['BALANCE_GL', 'BALANCE_BANK', 'GL_BANK_VARIANCE', 'PERIOD_ABS_VARIANCE',
                        'GL_BALANCE_ZSCORE', 'VARIANCE_ZSCORE', 'ABS_VARIANCE_ZSCORE',
                        'VARIANCE_TO_GL_RATIO', 'GL_PERIOD_CHANGE', 'HIST_AVG_ABS_VARIANCE']

corr_matrix = feature_df[correlation_features].corr()

fig = px.imshow(corr_matrix, text_auto='.2f', aspect='auto',
                color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                title='Feature Correlation Matrix')
fig.update_layout(height=600, width=800)
fig.show()

---
## 2. Model Selection Decision

Based on EDA findings:
- **Class Imbalance**: Anomalies represent a small fraction (~10-15%) of data
- **High Dimensionality**: Multiple correlated features
- **Time-Series Nature**: Need to prevent data leakage

**Selected Models:**
1. **Isolation Forest** - Excellent for high-dimensional anomaly detection
2. **One-Class SVM** - Good for learning normal patterns
3. **Local Outlier Factor (LOF)** - Density-based approach
4. **XGBoost Classifier** - Supervised approach using labels

We'll use **time-based cross-validation** to prevent leakage.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score, 
                             confusion_matrix, classification_report, precision_recall_curve, average_precision_score)
from xgboost import XGBClassifier
import io
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print("ML libraries loaded successfully")

---
## 3. Feature Engineering & Train/Test Split

**Critical: Time-based split to prevent data leakage**

In [ ]:
feature_df_sorted = feature_df.sort_values(['PERIOD_YEAR', 'PERIOD_QUARTER', 'PERIOD_MONTH_NUM']).reset_index(drop=True)

ml_features = [
    'BALANCE_GL', 'BALANCE_BANK', 'GL_BANK_VARIANCE', 'PERIOD_ABS_VARIANCE',
    'BALANCE_GL_DIFF', 'BALANCE_BANK_DIFF',
    'HIST_AVG_GL_BALANCE', 'HIST_STDDEV_GL_BALANCE', 'HIST_AVG_VARIANCE',
    'HIST_STDDEV_VARIANCE', 'HIST_AVG_ABS_VARIANCE', 'HIST_STDDEV_ABS_VARIANCE',
    'GL_BALANCE_ZSCORE', 'VARIANCE_ZSCORE', 'ABS_VARIANCE_ZSCORE',
    'VARIANCE_TO_GL_RATIO', 'GL_PERIOD_CHANGE',
    'OWNERSHIP', 'HIERARCHY_DEPTH'
]

X = feature_df_sorted[ml_features].fillna(0)
y = feature_df_sorted['IS_ANOMALY_LABEL'].fillna(0)

split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Training set: {X_train.shape[0]} samples ({y_train.sum():.0f} anomalies, {100*y_train.mean():.1f}%)")
print(f"Test set: {X_test.shape[0]} samples ({y_test.sum():.0f} anomalies, {100*y_test.mean():.1f}%)")
print(f"\nFeatures used: {len(ml_features)}")
print(f"Time-based split: Train on earlier periods, test on later periods (no leakage)")

In [ ]:
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Scaled features - Train mean: {X_train_scaled.mean():.4f}, std: {X_train_scaled.std():.4f}")
print(f"Scaled features - Test mean: {X_test_scaled.mean():.4f}, std: {X_test_scaled.std():.4f}")

---
## 4. Experiment Tracking Setup

Using Snowflake ML Experiment Tracking

In [ ]:
from snowflake.ml.experiment import ExperimentTracking
from snowflake.ml.registry import Registry
import datetime

DB = 'COCO_LIVE_DB'
SCHEMA = 'PUBLIC'

exp = ExperimentTracking(session=session, database_name=DB, schema_name=SCHEMA)
exp.set_experiment('variance_anomaly_detection')

print(f"Experiment created: {DB}.{SCHEMA}.variance_anomaly_detection")

---
## 5. Model Training with Hyperparameter Tuning

### 5.1 Isolation Forest

In [ ]:
import time
run_id = int(time.time())

with exp.start_run(f'isolation_forest_{run_id}'):
    contamination_estimate = min(y_train.mean(), 0.4)
    
    param_grid_if = {
        'n_estimators': [100, 200],
        'max_samples': ['auto', 0.8],
        'contamination': [contamination_estimate, 0.1, 0.2],
        'max_features': [0.8, 1.0]
    }
    
    exp.log_params({
        'model_type': 'IsolationForest',
        'param_grid': str(param_grid_if),
        'cv_strategy': 'TimeSeriesSplit',
        'n_splits': 3
    })
    
    best_if_score = -np.inf
    best_if_model = None
    best_if_params = None
    
    for n_est in param_grid_if['n_estimators']:
        for max_samp in param_grid_if['max_samples']:
            for contam in param_grid_if['contamination']:
                for max_feat in param_grid_if['max_features']:
                    model = IsolationForest(
                        n_estimators=n_est,
                        max_samples=max_samp,
                        contamination=contam,
                        max_features=max_feat,
                        random_state=42,
                        n_jobs=-1
                    )
                    model.fit(X_train_scaled)
                    
                    y_pred_train = (model.predict(X_train_scaled) == -1).astype(int)
                    f1 = f1_score(y_train, y_pred_train, zero_division=0)
                    
                    if f1 > best_if_score:
                        best_if_score = f1
                        best_if_model = model
                        best_if_params = {'n_estimators': n_est, 'max_samples': max_samp, 
                                         'contamination': contam, 'max_features': max_feat}
    
    y_pred_if = (best_if_model.predict(X_test_scaled) == -1).astype(int)
    y_scores_if = -best_if_model.score_samples(X_test_scaled)
    
    precision_if = precision_score(y_test, y_pred_if, zero_division=0)
    recall_if = recall_score(y_test, y_pred_if, zero_division=0)
    f1_if = f1_score(y_test, y_pred_if, zero_division=0)
    
    if len(np.unique(y_test)) > 1:
        auc_if = roc_auc_score(y_test, y_scores_if)
        ap_if = average_precision_score(y_test, y_scores_if)
    else:
        auc_if = 0.0
        ap_if = 0.0
    
    exp.log_params(best_if_params)
    exp.log_metrics({
        'precision': precision_if,
        'recall': recall_if,
        'f1_score': f1_if,
        'roc_auc': auc_if,
        'avg_precision': ap_if
    })
    
    print(f"Isolation Forest Results:")
    print(f"  Best params: {best_if_params}")
    print(f"  Precision: {precision_if:.4f}")
    print(f"  Recall: {recall_if:.4f}")
    print(f"  F1 Score: {f1_if:.4f}")
    print(f"  ROC AUC: {auc_if:.4f}")

### 5.2 XGBoost Classifier (Supervised)

In [ ]:
with exp.start_run(f'xgboost_{run_id}'):
    tscv = TimeSeriesSplit(n_splits=3)
    
    param_grid_xgb = {
        'n_estimators': [100, 200],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1],
        'min_child_weight': [1, 3],
        'scale_pos_weight': [1, (y_train == 0).sum() / max((y_train == 1).sum(), 1)]
    }
    
    exp.log_params({
        'model_type': 'XGBClassifier',
        'cv_strategy': 'TimeSeriesSplit',
        'n_splits': 3
    })
    
    base_xgb = XGBClassifier(
        objective='binary:logistic',
        eval_metric='aucpr',
        random_state=42,
        use_label_encoder=False
    )
    
    grid_search = GridSearchCV(
        base_xgb,
        param_grid_xgb,
        cv=tscv,
        scoring='f1',
        n_jobs=-1,
        verbose=0
    )
    
    grid_search.fit(X_train_scaled, y_train)
    best_xgb_model = grid_search.best_estimator_
    
    y_pred_xgb = best_xgb_model.predict(X_test_scaled)
    y_proba_xgb = best_xgb_model.predict_proba(X_test_scaled)[:, 1]
    
    precision_xgb = precision_score(y_test, y_pred_xgb, zero_division=0)
    recall_xgb = recall_score(y_test, y_pred_xgb, zero_division=0)
    f1_xgb = f1_score(y_test, y_pred_xgb, zero_division=0)
    
    if len(np.unique(y_test)) > 1:
        auc_xgb = roc_auc_score(y_test, y_proba_xgb)
        ap_xgb = average_precision_score(y_test, y_proba_xgb)
    else:
        auc_xgb = 0.0
        ap_xgb = 0.0
    
    exp.log_params(grid_search.best_params_)
    exp.log_metrics({
        'precision': precision_xgb,
        'recall': recall_xgb,
        'f1_score': f1_xgb,
        'roc_auc': auc_xgb,
        'avg_precision': ap_xgb,
        'cv_best_score': grid_search.best_score_
    })
    
    print(f"XGBoost Results:")
    print(f"  Best params: {grid_search.best_params_}")
    print(f"  Precision: {precision_xgb:.4f}")
    print(f"  Recall: {recall_xgb:.4f}")
    print(f"  F1 Score: {f1_xgb:.4f}")
    print(f"  ROC AUC: {auc_xgb:.4f}")
    print(f"  CV Best Score: {grid_search.best_score_:.4f}")

### 5.3 One-Class SVM

In [ ]:
with exp.start_run(f'ocsvm_{run_id}'):
    X_train_normal = X_train_scaled[y_train == 0]
    
    param_grid_svm = {
        'kernel': ['rbf'],
        'gamma': ['scale', 'auto', 0.1],
        'nu': [0.05, 0.1, 0.15]
    }
    
    exp.log_params({
        'model_type': 'OneClassSVM',
        'training_strategy': 'normal_samples_only'
    })
    
    best_svm_score = -np.inf
    best_svm_model = None
    best_svm_params = None
    
    for kernel in param_grid_svm['kernel']:
        for gamma in param_grid_svm['gamma']:
            for nu in param_grid_svm['nu']:
                model = OneClassSVM(kernel=kernel, gamma=gamma, nu=nu)
                model.fit(X_train_normal[:10000])
                
                y_pred_train = (model.predict(X_train_scaled) == -1).astype(int)
                f1 = f1_score(y_train, y_pred_train, zero_division=0)
                
                if f1 > best_svm_score:
                    best_svm_score = f1
                    best_svm_model = model
                    best_svm_params = {'kernel': kernel, 'gamma': gamma, 'nu': nu}
    
    y_pred_svm = (best_svm_model.predict(X_test_scaled) == -1).astype(int)
    y_scores_svm = -best_svm_model.decision_function(X_test_scaled)
    
    precision_svm = precision_score(y_test, y_pred_svm, zero_division=0)
    recall_svm = recall_score(y_test, y_pred_svm, zero_division=0)
    f1_svm = f1_score(y_test, y_pred_svm, zero_division=0)
    
    if len(np.unique(y_test)) > 1:
        auc_svm = roc_auc_score(y_test, y_scores_svm)
    else:
        auc_svm = 0.0
    
    exp.log_params(best_svm_params)
    exp.log_metrics({
        'precision': precision_svm,
        'recall': recall_svm,
        'f1_score': f1_svm,
        'roc_auc': auc_svm
    })
    
    print(f"One-Class SVM Results:")
    print(f"  Best params: {best_svm_params}")
    print(f"  Precision: {precision_svm:.4f}")
    print(f"  Recall: {recall_svm:.4f}")
    print(f"  F1 Score: {f1_svm:.4f}")
    print(f"  ROC AUC: {auc_svm:.4f}")

---
## 6. Model Comparison & Visualization

In [ ]:
results_df = pd.DataFrame({
    'Model': ['Isolation Forest', 'XGBoost', 'One-Class SVM'],
    'Precision': [precision_if, precision_xgb, precision_svm],
    'Recall': [recall_if, recall_xgb, recall_svm],
    'F1 Score': [f1_if, f1_xgb, f1_svm],
    'ROC AUC': [auc_if, auc_xgb, auc_svm]
})

print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)
print(results_df.to_string(index=False))
print("="*60)

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Model Performance Metrics', 'F1 Score Comparison'))

metrics = ['Precision', 'Recall', 'F1 Score', 'ROC AUC']
colors = ['#3498db', '#e74c3c', '#2ecc71']

for i, model in enumerate(results_df['Model']):
    fig.add_trace(go.Bar(
        name=model,
        x=metrics,
        y=results_df.iloc[i][metrics].values,
        marker_color=colors[i]
    ), row=1, col=1)

fig.add_trace(go.Bar(
    x=results_df['Model'],
    y=results_df['F1 Score'],
    marker_color=colors,
    showlegend=False
), row=1, col=2)

fig.update_layout(height=400, title_text='Model Performance Comparison', barmode='group')
fig.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

if len(np.unique(y_test)) > 1:
    prec_if, rec_if, _ = precision_recall_curve(y_test, y_scores_if)
    prec_xgb, rec_xgb, _ = precision_recall_curve(y_test, y_proba_xgb)
    prec_svm, rec_svm, _ = precision_recall_curve(y_test, y_scores_svm)
    
    ax.plot(rec_if, prec_if, label=f'Isolation Forest (AP={ap_if:.3f})', linewidth=2)
    ax.plot(rec_xgb, prec_xgb, label=f'XGBoost (AP={ap_xgb:.3f})', linewidth=2)
    ax.plot(rec_svm, prec_svm, label=f'One-Class SVM', linewidth=2)
    
    ax.set_xlabel('Recall', fontsize=12)
    ax.set_ylabel('Precision', fontsize=12)
    ax.set_title('Precision-Recall Curves', fontsize=14)
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

buf = io.BytesIO()
plt.savefig(buf, format='png', dpi=150, bbox_inches='tight')
buf.seek(0)
plt.close()

print("Precision-Recall curves generated")

---
## 7. Model Registry - Register Best Model

In [ ]:
best_model_idx = results_df['F1 Score'].idxmax()
best_model_name = results_df.loc[best_model_idx, 'Model']
best_f1 = results_df.loc[best_model_idx, 'F1 Score']

print(f"Best performing model: {best_model_name}")
print(f"F1 Score: {best_f1:.4f}")

if best_model_name == 'Isolation Forest':
    model_to_register = best_if_model
elif best_model_name == 'XGBoost':
    model_to_register = best_xgb_model
else:
    model_to_register = best_svm_model

In [ ]:
registry = Registry(session=session, database_name=DB, schema_name=SCHEMA)

sample_input = pd.DataFrame(X_test_scaled[:5], columns=ml_features)

model_name = 'variance_anomaly_detector'
version_name = f'v1_{best_model_name.lower().replace(" ", "_").replace("-", "_")}'

try:
    mv = registry.log_model(
        model=model_to_register,
        model_name=model_name,
        version_name=version_name,
        sample_input_data=sample_input,
        metrics={
            'f1_score': float(best_f1),
            'precision': float(results_df.loc[best_model_idx, 'Precision']),
            'recall': float(results_df.loc[best_model_idx, 'Recall']),
            'roc_auc': float(results_df.loc[best_model_idx, 'ROC AUC'])
        },
        comment=f'Variance anomaly detection model - {best_model_name}'
    )
    print(f"Model registered: {model_name}/{version_name}")
    print(f"Metrics logged: F1={best_f1:.4f}")
except Exception as e:
    print(f"Registration note: {e}")

---
## 8. Version Comparison & Best Model Promotion

In [ ]:
with exp.start_run(f'xgboost_v2_{run_id}'):
    param_grid_v2 = {
        'n_estimators': [300],
        'max_depth': [5, 7, 9],
        'learning_rate': [0.05, 0.1],
        'min_child_weight': [1, 5],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0],
        'scale_pos_weight': [(y_train == 0).sum() / max((y_train == 1).sum(), 1)]
    }
    
    exp.log_params({'model_type': 'XGBClassifier_v2', 'tuning': 'extended'})
    
    base_xgb_v2 = XGBClassifier(
        objective='binary:logistic',
        eval_metric='aucpr',
        random_state=42,
        use_label_encoder=False
    )
    
    random_search = RandomizedSearchCV(
        base_xgb_v2,
        param_grid_v2,
        n_iter=20,
        cv=TimeSeriesSplit(n_splits=3),
        scoring='f1',
        n_jobs=-1,
        random_state=42,
        verbose=0
    )
    
    random_search.fit(X_train_scaled, y_train)
    best_xgb_v2 = random_search.best_estimator_
    
    y_pred_v2 = best_xgb_v2.predict(X_test_scaled)
    y_proba_v2 = best_xgb_v2.predict_proba(X_test_scaled)[:, 1]
    
    precision_v2 = precision_score(y_test, y_pred_v2, zero_division=0)
    recall_v2 = recall_score(y_test, y_pred_v2, zero_division=0)
    f1_v2 = f1_score(y_test, y_pred_v2, zero_division=0)
    auc_v2 = roc_auc_score(y_test, y_proba_v2) if len(np.unique(y_test)) > 1 else 0
    
    exp.log_params(random_search.best_params_)
    exp.log_metrics({
        'precision': precision_v2,
        'recall': recall_v2,
        'f1_score': f1_v2,
        'roc_auc': auc_v2
    })
    
    print(f"XGBoost V2 Results:")
    print(f"  F1 Score: {f1_v2:.4f} (vs V1: {f1_xgb:.4f})")
    print(f"  Improvement: {(f1_v2 - f1_xgb)*100:.2f}%")

In [ ]:
all_versions = [
    ('v1_isolation_forest', f1_if, best_if_model),
    ('v1_xgboost', f1_xgb, best_xgb_model),
    ('v1_ocsvm', f1_svm, best_svm_model),
    ('v2_xgboost_tuned', f1_v2, best_xgb_v2)
]

version_comparison = pd.DataFrame([
    {'Version': v[0], 'F1 Score': v[1]} for v in all_versions
]).sort_values('F1 Score', ascending=False)

print("\n" + "="*50)
print("ALL MODEL VERSIONS - RANKED BY F1 SCORE")
print("="*50)
print(version_comparison.to_string(index=False))
print("="*50)

best_version = version_comparison.iloc[0]
print(f"\nBest Version: {best_version['Version']}")
print(f"F1 Score: {best_version['F1 Score']:.4f}")

In [ ]:
PROMOTION_THRESHOLD = 0.01

current_best_f1 = best_f1
new_best_version = version_comparison.iloc[0]
new_best_f1 = new_best_version['F1 Score']
new_best_name = new_best_version['Version']

improvement = new_best_f1 - current_best_f1

print(f"\n{'='*60}")
print("MODEL PROMOTION DECISION")
print(f"{'='*60}")
print(f"Current default model F1: {current_best_f1:.4f}")
print(f"Best candidate F1: {new_best_f1:.4f}")
print(f"Improvement: {improvement*100:.2f}%")
print(f"Promotion threshold: {PROMOTION_THRESHOLD*100:.1f}%")
print(f"{'='*60}")

if improvement > PROMOTION_THRESHOLD:
    print(f"\nPROMOTING: {new_best_name} as new default")
    print("Reason: Exceeds improvement threshold")
    
    for v in all_versions:
        if v[0] == new_best_name:
            promoted_model = v[2]
            break
    
    try:
        mv_promoted = registry.log_model(
            model=promoted_model,
            model_name=model_name,
            version_name=new_best_name,
            sample_input_data=sample_input,
            metrics={'f1_score': float(new_best_f1)},
            comment=f'Promoted as best model - F1: {new_best_f1:.4f}'
        )
        
        model_ref = registry.get_model(model_name)
        model_ref.default = new_best_name
        print(f"Default version set to: {new_best_name}")
    except Exception as e:
        print(f"Promotion note: {e}")
else:
    print(f"\nKEEPING current default model")
    print(f"Reason: Improvement ({improvement*100:.2f}%) below threshold ({PROMOTION_THRESHOLD*100:.1f}%)")

---
## 9. Summary & Next Steps

In [ ]:
print("\n" + "="*70)
print("ANOMALY DETECTION ML PIPELINE - SUMMARY")
print("="*70)
print(f"\nDataset: {len(feature_df):,} samples from MART_ANOMALY_FEATURES")
print(f"Features: {len(ml_features)} engineered features")
print(f"Train/Test Split: 80/20 time-based (no data leakage)")
print(f"\nModels Trained:")
print(f"  - Isolation Forest (unsupervised)")
print(f"  - XGBoost Classifier (supervised)")
print(f"  - One-Class SVM (semi-supervised)")
print(f"  - XGBoost V2 Tuned (extended HPO)")
print(f"\nBest Model: {version_comparison.iloc[0]['Version']}")
print(f"Best F1 Score: {version_comparison.iloc[0]['F1 Score']:.4f}")
print(f"\nArtifacts Created:")
print(f"  - Experiment: {DB}.{SCHEMA}.variance_anomaly_detection")
print(f"  - Model Registry: {DB}.{SCHEMA}.{model_name}")
print(f"  - Feature Store: {DB}.{SCHEMA}.MART_ANOMALY_FEATURES")
print("="*70)
print("\nNEXT STEPS:")
print("  1. Deploy model for batch inference on new reconciliations")
print("  2. Set up model monitoring for drift detection")
print("  3. Create alerts for high-confidence anomaly predictions")
print("  4. Integrate predictions into Entity 360 dashboard")
print("="*70)